# Build the tech-jobs index (v1)

LinkedIn postings (CC BY-SA 4.0) → ~15k tech jobs → BM25 + `bge-small` embeddings + skills from `srish327/deberta-v3-skillspan-crf`
→ HF dataset `srish327/tech-jobs-index/v1`.

**Needs:** input dataset `arshkon/linkedin-job-postings`, GPU, Internet on, and secret `hg` (HF write token) attached.

In [ ]:
%pip install -q "jobmatch[index] @ git+https://github.com/MSR327/resume-job-matcher.git@d5f407b"
BUILD_COMMIT = "d5f407b"

In [ ]:
# Fail fast: check the HF token and write access now, not after 30 minutes of GPU work.
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hg")  # also authenticates model downloads
api = HfApi()
REPO = f"{api.whoami()['name']}/tech-jobs-index"
api.create_repo(REPO, repo_type="dataset", exist_ok=True)  # raises now if the token cannot write
print("HF auth ok ->", REPO)

In [ ]:
import glob

import pandas as pd

from jobmatch.corpus import build_corpus

path = [p for p in glob.glob("/kaggle/input/**/postings.csv", recursive=True)][0]
df = pd.read_csv(path)
print(path, df.shape)
need = {"job_id", "title", "company_name", "location", "description", "job_posting_url", "listed_time"}
assert need <= set(df.columns), need - set(df.columns)
jobs = build_corpus(df)
print("tech jobs:", len(jobs))
print(jobs["title"].value_counts().head(25).to_string())
print(jobs["description"].str.len().describe(percentiles=[.05, .5, .95]).round(0).to_string())
assert 5_000 <= len(jobs) <= 40_000, "corpus size outside expected range; check the filter before building"

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import time

from skillex.predict import SkillExtractor

dev = "cuda" if torch.cuda.is_available() else "cpu"
emb = SentenceTransformer("BAAI/bge-small-en-v1.5", device=dev)
ext = SkillExtractor("srish327/deberta-v3-skillspan-crf", device=dev)


def embed_docs(texts):
    return emb.encode(texts, batch_size=128, normalize_embeddings=True, show_progress_bar=False)


def extract_skills(descs):
    out, t0 = [], time.time()
    for i, d in enumerate(descs, 1):
        out.append([{"type": s["type"], "text": s["text"]} for s in ext.extract(d)["spans"]])
        if i % 500 == 0 or i == len(descs):
            print(f"skills {i}/{len(descs)}  {(time.time() - t0) / 60:.1f} min", flush=True)
    return out

In [ ]:
from pathlib import Path

from jobmatch.index import build_index

idx = build_index(jobs, embed_docs, extract_skills, version="v1", extra_manifest={
    "embed_model": "BAAI/bge-small-en-v1.5", "skill_model": "srish327/deberta-v3-skillspan-crf",
    "source": "kaggle:arshkon/linkedin-job-postings (CC BY-SA 4.0)", "build_commit": BUILD_COMMIT})
idx.save("artifacts/v1")
total = sum(f.stat().st_size for f in Path("artifacts/v1").rglob("*") if f.is_file())
print(idx.manifest, f"\nsize: {total/1e6:.1f} MB")
n_sk = [len(s) for s in idx.skills]
print("raw spans", idx.manifest["n_spans_raw"], "-> kept", idx.manifest["n_spans_kept"],
      "| distinct phrases", idx.manifest["n_phrases"])
print("skills per job: mean", round(sum(n_sk) / len(n_sk), 1), "| jobs with 0 skills:", sum(n == 0 for n in n_sk))
for i in (0, len(jobs) // 2, len(jobs) - 1):
    print("\n", jobs.loc[i, "title"], "|", [s["text"] for s in idx.skills[i]][:15])
if total > 250e6:  # warn, never throw away a finished build
    print(f"WARNING: index is {total/1e6:.0f} MB; the app needs it small (Streamlit has 1 GB RAM)")

In [ ]:
!curl -sL https://raw.githubusercontent.com/MSR327/resume-job-matcher/main/DATA_CARD.md -o DATA_CARD.md
api.upload_file(path_or_fileobj="DATA_CARD.md", path_in_repo="README.md", repo_id=REPO, repo_type="dataset")
api.upload_folder(folder_path="artifacts/v1", path_in_repo="v1", repo_id=REPO, repo_type="dataset",
                  commit_message=f"v1: {len(jobs)} tech jobs (build {BUILD_COMMIT})")
print("https://huggingface.co/datasets/" + REPO)